# Sprint 6 – Notebook 15: Complete Feature Engineering Workflow

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier

# Synthetic customer and transaction records (Sprint 5-style sample data).
rng = np.random.default_rng(42)
n = 120
customers = np.arange(1001, 1031)
df = pd.DataFrame({
    "Customer_ID": rng.choice(customers, n),
    "Age": rng.integers(20, 65, n),
    "Annual_Income": rng.integers(250000, 1500000, n),
    "City": rng.choice(["Hassan", "Bengaluru", "Mysuru"], n),
    "Membership_Type": rng.choice(["Basic", "Silver", "Gold"], n),
    "Product_Category": rng.choice(["Electronics", "Clothing", "Grocery"], n),
    "Product_Price": rng.integers(200, 9000, n),
    "Quantity": rng.integers(1, 5, n),
    "Discount_Percent": rng.choice([0, 5, 10, 15], n),
    "Purchase_Date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 300, n), unit="D"),
    "Review_Text": rng.choice(["Great quality", "Good value", "Late delivery", "Very happy"], n),
})
df["Repeat_Purchase"] = ((df["Membership_Type"] == "Gold") | (df["Quantity"] >= 3)).astype(int)
print("Clean sample dataset:", df.shape)
display(df.head(3))

Clean sample dataset: (120, 12)


,Customer_ID,Age,Annual_Income,City,Membership_Type,Product_Category,Product_Price,Quantity,Discount_Percent,Purchase_Date,Review_Text,Repeat_Purchase
0,1003,39,425280,Mysuru,Basic,Clothing,1424,2,0,2025-10-04,Very happy,0
1,1024,50,980122,Mysuru,Gold,Grocery,8394,3,5,2025-10-24,Good value,1
2,1020,49,368368,Hassan,Basic,Grocery,7636,4,15,2025-10-15,Very happy,1


## Topic 1 – Load Clean Dataset

### 1. Definition
Loading data brings cleaned customer records into a DataFrame. Here the synthetic sample is created in the first cell.

### 2. Business Example
A retailer collects purchase records for analysis.

### 3. AI/ML Use
Data must be loaded consistently before features are engineered.

### 4. Python Example

In [4]:
print("Dataset shape:", df.shape)
display(df.head(3))

Dataset shape: (120, 12)


,Customer_ID,Age,Annual_Income,City,Membership_Type,Product_Category,Product_Price,Quantity,Discount_Percent,Purchase_Date,Review_Text,Repeat_Purchase
0,1003,39,425280,Mysuru,Basic,Clothing,1424,2,0,2025-10-04,Very happy,0
1,1024,50,980122,Mysuru,Gold,Grocery,8394,3,5,2025-10-24,Good value,1
2,1020,49,368368,Hassan,Basic,Grocery,7636,4,15,2025-10-15,Very happy,1


### 5. Explanation
The DataFrame has customer, product, purchase date, and target columns. The same base data is reused throughout the notebook.

## Topic 2 – Understand Existing Features

### 1. Definition
Review the columns and types to understand which inputs are available. Check whether dates and identifiers need special handling.

### 2. Business Example
A store checks which customer and order fields are available.

### 3. AI/ML Use
Identifying feature types guides suitable transformations.

### 4. Python Example

In [7]:
print(df.dtypes)
print("Target distribution:\n", df["Repeat_Purchase"].value_counts())

Customer_ID                  int32
Age                          int64
Annual_Income                int64
City                        object
Membership_Type             object
Product_Category            object
Product_Price                int64
Quantity                     int64
Discount_Percent             int32
Purchase_Date       datetime64[ns]
Review_Text                 object
Repeat_Purchase              int32
dtype: object
Target distribution:
 Repeat_Purchase
1    84
0    36
Name: count, dtype: int64


### 5. Explanation
Age and income are numeric, while city and membership are categorical. Purchase_Date is a datetime column; Repeat_Purchase is the target.

## Topic 3 – Identify Potential Feature Gaps

### 1. Definition
Feature gaps are useful signals not directly stored in raw columns. New features should use information available at prediction time.

### 2. Business Example
The store has price and quantity but no final order amount.

### 3. AI/ML Use
New meaningful features can improve a model without collecting new raw data.

### 4. Python Example

In [10]:
print("Potential features: order value, discount amount, age band, purchase month, text length")

Potential features: order value, discount amount, age band, purchase month, text length


### 5. Explanation
The listed features can be derived from existing columns. Future purchase counts would be unsafe for predicting repeat purchases at an earlier time.

## Topic 4 – Create Numerical Features

### 1. Definition
Numerical feature creation combines measured values into useful quantities. Arithmetic and ratios should have clear business meaning.

### 2. Business Example
A retailer calculates gross and discounted order value.

### 3. AI/ML Use
Derived values capture purchasing behavior better than raw price alone.

### 4. Python Example

In [13]:
engineered = df.copy()
engineered["Gross_Order_Value"] = engineered["Product_Price"] * engineered["Quantity"]
engineered["Net_Order_Value"] = engineered["Gross_Order_Value"] * (1 - engineered["Discount_Percent"] / 100)
print(engineered[["Gross_Order_Value", "Net_Order_Value"]].head())

   Gross_Order_Value  Net_Order_Value
0               2848           2848.0
1              25182          23922.9
2              30544          25962.4
3               1253           1253.0
4               7116           6048.6


### 5. Explanation
Gross value multiplies price and quantity. Net value accounts for discount, providing a direct spending measure.

## Topic 5 – Create Categorical Features

### 1. Definition
Categorical features group continuous values or combine existing labels. Categories should be interpretable and not based on the target.

### 2. Business Example
The store groups customers into age segments.

### 3. AI/ML Use
Age bands can capture non-linear patterns and support reporting.

### 4. Python Example

In [16]:
engineered["Age_Group"] = pd.cut(engineered["Age"], bins=[0, 30, 45, 100], labels=["Young", "Adult", "Senior"])
print(engineered["Age_Group"].value_counts())

Age_Group
Adult     48
Senior    48
Young     24
Name: count, dtype: int64


### 5. Explanation
Age groups simplify customer segmentation. The bin boundaries are illustrative business rules.

## Topic 6 – Create Date/Time Features

### 1. Definition
Date features extract calendar information from timestamps. They can represent seasonality and shopping patterns.

### 2. Business Example
A retailer tracks weekend and month-specific demand.

### 3. AI/ML Use
Calendar indicators may help models learn when purchasing changes.

### 4. Python Example

In [19]:
engineered["Purchase_Month"] = engineered["Purchase_Date"].dt.month
engineered["Is_Weekend"] = (engineered["Purchase_Date"].dt.dayofweek >= 5).astype(int)
print(engineered[["Purchase_Month", "Is_Weekend"]].head())

   Purchase_Month  Is_Weekend
0              10           1
1              10           0
2              10           0
3               6           0
4               3           1


### 5. Explanation
The month is an integer from 1 to 12. The weekend indicator is 1 on Saturday and Sunday.

## Topic 7 – Create Aggregation Features

### 1. Definition
Aggregations summarize multiple transactions for each customer. They must be calculated from past observations only when used for prediction.

### 2. Business Example
The retailer counts earlier purchases by each customer.

### 3. AI/ML Use
Historical activity can reveal loyalty without using future information.

### 4. Python Example

In [22]:
engineered = engineered.sort_values(["Purchase_Date", "Customer_ID"]).copy()
engineered["Prior_Order_Count"] = engineered.groupby("Customer_ID").cumcount()
engineered["Prior_Spending"] = engineered.groupby("Customer_ID")["Net_Order_Value"].transform(lambda s: s.cumsum() - s)
print(engineered[["Customer_ID", "Prior_Order_Count", "Prior_Spending"]].head())

     Customer_ID  Prior_Order_Count  Prior_Spending
109         1022                  0             0.0
94          1027                  0             0.0
40          1005                  0             0.0
86          1003                  0             0.0
99          1005                  1          1485.8


### 5. Explanation
cumcount counts earlier rows per customer. Cumulative spending subtracts the current order; in production, ties and event-time availability require careful handling.

## Topic 8 – Create Interaction Features

### 1. Definition
Interaction features combine inputs to represent joint effects. They should have a sensible interpretation.

### 2. Business Example
A retailer measures how large an order is relative to customer income.

### 3. AI/ML Use
Interactions can reveal effects not visible in either feature alone.

### 4. Python Example

In [25]:
engineered["Order_Income_Ratio"] = engineered["Net_Order_Value"] / engineered["Annual_Income"].clip(lower=1)
engineered["Review_Length"] = engineered["Review_Text"].str.len()
print(engineered[["Order_Income_Ratio", "Review_Length"]].head())

     Order_Income_Ratio  Review_Length
109            0.001997             10
94             0.017080             10
40             0.002498             13
86             0.027467             10
99             0.017783             13


### 5. Explanation
The ratio normalizes order value by income. Review length is a simple text feature, but must be excluded if reviews arrive after prediction.

## Topic 9 – Perform Feature Selection

### 1. Definition
Feature selection keeps useful predictors and removes irrelevant ones. Selection must be fitted on training data only.

### 2. Business Example
A retailer compares many engineered variables before model training.

### 3. AI/ML Use
Reducing features may improve stability and simplify deployment.

### 4. Python Example

In [28]:
X = engineered.drop(columns=["Repeat_Purchase", "Customer_ID", "Purchase_Date", "Review_Text", "Review_Length"])
y = engineered["Repeat_Purchase"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (90, 16) Test: (30, 16)


### 5. Explanation
Customer IDs are identifiers, not predictors. Review features are excluded because their availability at purchase time is uncertain.

## Topic 10 – Analyze Feature Importance

### 1. Definition
Feature importance estimates how much a fitted model relies on each input. Importance is not evidence of causation.

### 2. Business Example
A retailer checks which purchase features influence predictions.

### 3. AI/ML Use
Importance can help explain model behavior and guide future investigation.

### 4. Python Example

In [31]:
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
preprocessor = ColumnTransformer([("num", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols), ("cat", Pipeline([("fill", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)])
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
model = RandomForestClassifier(n_estimators=60, random_state=42).fit(X_train_encoded, y_train)
importance = pd.Series(model.feature_importances_, index=preprocessor.get_feature_names_out()).sort_values(ascending=False)
print(importance.head(6))

num__Quantity                0.218519
cat__Membership_Type_Gold    0.153564
num__Order_Income_Ratio      0.094678
num__Gross_Order_Value       0.068809
num__Net_Order_Value         0.064927
num__Product_Price           0.058367
dtype: float64


### 5. Explanation
Preprocessing is fitted on training records. The random forest ranks transformed columns; synthetic targets can make some features artificially strong.

## Topic 11 – Check for Feature Leakage

### 1. Definition
Leakage happens when predictors contain unavailable future or target information. Check timing, provenance, and train/test separation.

### 2. Business Example
A customer review may be written after the purchase.

### 3. AI/ML Use
Leakage can produce misleadingly high evaluation scores.

### 4. Python Example

In [34]:
forbidden = {"Repeat_Purchase", "Customer_ID", "Purchase_Date", "Review_Text", "Review_Length"}
print("Forbidden predictors in X:", sorted(forbidden.intersection(X.columns)))
print("Preprocessor fitted on training only:", True)

Forbidden predictors in X: []
Preprocessor fitted on training only: True


### 5. Explanation
The forbidden columns are absent from model inputs. This check is necessary but not sufficient: historical aggregates must also respect event timestamps.

## Topic 12 – Remove Unnecessary Features

### 1. Definition
Unnecessary features may be IDs, duplicates, unavailable fields, or low-value predictors. Removing them reduces complexity.

### 2. Business Example
The retailer removes raw identifiers and unverified post-purchase text.

### 3. AI/ML Use
A smaller feature set can be easier to maintain and audit.

### 4. Python Example

In [37]:
selector = SelectKBest(score_func=f_classif, k=min(8, X_train_encoded.shape[1]))
X_train_selected = selector.fit_transform(X_train_encoded, y_train)
X_test_selected = selector.transform(X_test_encoded)
print("Selected features:", X_train_selected.shape[1])

Selected features: 8


### 5. Explanation
The selection method uses training labels only. The same learned selection is applied unchanged to test records.

## Topic 13 – Create Final Feature Dataset

### 1. Definition
An ML-ready feature set contains numeric columns with consistent train/test structure. Keep the target separate from predictors.

### 2. Business Example
The retailer prepares input matrices for a classification model.

### 3. AI/ML Use
Clean, selected matrices can be passed directly into ML algorithms.

### 4. Python Example

In [40]:
selected_names = preprocessor.get_feature_names_out()[selector.get_support()]
final_train = pd.DataFrame(X_train_selected, columns=selected_names, index=X_train.index)
final_test = pd.DataFrame(X_test_selected, columns=selected_names, index=X_test.index)
print("Final train:", final_train.shape, "Final test:", final_test.shape)
print("Missing in final train:", final_train.isna().sum().sum())

Final train: (90, 8) Final test: (30, 8)
Missing in final train: 0


### 5. Explanation
Both datasets contain the same selected numeric features. Targets remain in y_train and y_test.

## Topic 14 – Document All Feature Engineering Decisions

### 1. Definition
Documentation records formulas, assumptions, timing, and removal decisions. It makes features reproducible and auditable.

### 2. Business Example
The analytics team records why a feature was included.

### 3. AI/ML Use
A clear feature dictionary supports review and deployment.

### 4. Python Example

In [43]:
decisions = {"Gross_Order_Value": "Price × Quantity", "Net_Order_Value": "Gross value after discount", "Prior_Order_Count": "Past orders only", "Review_Text": "Excluded: post-purchase timing uncertain", "Selection": "SelectKBest fitted on training only"}
for name, decision in decisions.items(): print(name, "→", decision)

Gross_Order_Value → Price × Quantity
Net_Order_Value → Gross value after discount
Prior_Order_Count → Past orders only
Review_Text → Excluded: post-purchase timing uncertain
Selection → SelectKBest fitted on training only


### 5. Explanation
The feature dictionary explains each important choice. Before production, confirm data availability and apply time-aware validation for transaction histories.

## Final Outcome
**Clean Dataset → Engineered Dataset → Selected Features → Final ML-Ready Feature Set**

`final_train` and `final_test` contain selected numerical ML inputs. `y_train` and `y_test` contain targets. This demonstration uses random splitting for convenience; use a time-based split and point-in-time aggregation in real transactional prediction tasks.